# OthelloRL Sweep Agent
Connects to the existing wandb sweep and runs trials on Colab hardware.

In [ ]:
# 1. Clone the repo
!git clone https://github.com/sanasuri101/OthelloRL.git
%cd OthelloRL

In [ ]:
# 2. Install dependencies
# Force-reinstall wandb to ensure wandb-core binary is present (avoids version conflicts)
!pip install -q --force-reinstall wandb
!pip install -q torch --index-url https://download.pytorch.org/whl/cu128
!pip install -q gymnasium==0.29.1 numpy==1.26.4
!pip install -q setuptools wheel
!pip install -q pufferlib --no-build-isolation

# Build the C extension without raylib (no rendering needed for training)
%cd othello
!make NO_RENDER=1
%cd ..

In [ ]:
# 3. Verify everything imports correctly
import sys, os
sys.path.insert(0, '/content/OthelloRL')
# Remove othello subdir from path to avoid relative import conflicts
sys.path = [p for p in sys.path if not p.endswith('/othello')]

import torch
import pufferlib
import wandb
print(f'torch: {torch.__version__}')
print(f'pufferlib: {pufferlib.__version__}')
print(f'wandb: {wandb.__version__}')
from othello import binding
print('binding: OK')

In [ ]:
# 4. Log in to wandb using Colab secret
import os
from google.colab import userdata

os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
os.environ["WANDB_AGENT_DISABLE_FLAPPING"] = "true"
os.chdir('/content/OthelloRL')

wandb.login()

In [ ]:
# 5. Create a new sweep (run only once — skip if you already have a sweep ID)
!wandb sweep /content/OthelloRL/sweep.yaml

In [ ]:
# 6. Run sweep trials in-process (no subprocess spawning — avoids all env var conflicts)
import sys

# Set argv so train()'s argparse gets the fixed sweep args
sys.argv = [
    'train.py',
    '--wandb',
    '--wandb_project', 'Connect4RL-othello',
    '--train.total_timesteps', '10000000',
    '--train.eval_interval', '500000',
    '--curriculum.phase_4_fraction', '0.60',
    '--curriculum.phase_5_fraction', '0.60',
]

from othello.train import train

SWEEP_ID = "sriramanasuri-georgia-institute-of-technology/Connect4RL-othello/ux3575ja"

def run_trial():
    train()

wandb.agent(
    sweep_id=SWEEP_ID,
    function=run_trial,
    count=10,
)